# SCBE-AETHERMOORE Free Fine-Tuning on Colab

Fine-tune a model on your 14,654 SFT training pairs using QLoRA.
**Cost: $0** — runs on Colab's free T4 GPU.

Dataset: `issdandavis/scbe-aethermoore-training-data`

## Setup
1. Open this notebook in Google Colab
2. Runtime → Change runtime type → T4 GPU
3. Run all cells

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_hub

In [ ]:
# Login to HuggingFace (paste your token)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================
# CONFIG — Change these to customize training
# ============================================

# Base model (choose one):
# - "microsoft/phi-2" (2.7B, fast, good for T4)
# - "TinyLlama/TinyLlama-1.1B-Chat-v1.0" (1.1B, fastest)
# - "mistralai/Mistral-7B-Instruct-v0.3" (7B, best quality, tight on T4)
# - "google/gemma-2b-it" (2B, good balance)
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Your dataset on HuggingFace
DATASET_ID = "issdandavis/scbe-aethermoore-training-data"

# Output model name (will be pushed to your HF account)
OUTPUT_MODEL = "issdandavis/scbe-aethermoore-sft-v1"

# Training config
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32

print(f"Base model: {BASE_MODEL}")
print(f"Dataset: {DATASET_ID}")
print(f"Output: {OUTPUT_MODEL}")

In [ ]:
# Load dataset
dataset = load_dataset(DATASET_ID, split="train")
print(f"Dataset size: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample:")
print(dataset[0])

In [ ]:
# Format dataset for SFT training
def format_for_sft(example):
    """Convert SCBE training pair to chat format."""
    # Handle different possible column formats
    if 'input' in example and 'output' in example:
        user_text = example['input']
        assistant_text = example['output']
    elif 'prompt' in example and 'response' in example:
        user_text = example['prompt']
        assistant_text = example['response']
    elif 'question' in example and 'answer' in example:
        user_text = example['question']
        assistant_text = example['answer']
    elif 'text' in example:
        return {"text": example['text']}  # Already formatted
    else:
        # Try to extract from nested dicts
        inp = example.get('input', {})
        out = example.get('output', {})
        if isinstance(inp, dict):
            user_text = inp.get('text', str(inp))
        else:
            user_text = str(inp)
        if isinstance(out, dict):
            assistant_text = out.get('text', str(out))
        else:
            assistant_text = str(out)

    # Format as chat
    text = (
        f"<|system|>\n"
        f"You are an AI assistant powered by SCBE-AETHERMOORE governance. "
        f"You are helpful, safe, and technically competent.</s>\n"
        f"<|user|>\n{user_text}</s>\n"
        f"<|assistant|>\n{assistant_text}</s>"
    )
    return {"text": text}

formatted = dataset.map(format_for_sft, remove_columns=dataset.column_names)
print(f"Formatted {len(formatted)} examples")
print(f"\nSample formatted text (first 500 chars):")
print(formatted[0]['text'][:500])

In [ ]:
# Load base model with 4-bit quantization (fits on free T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare for QLoRA
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {BASE_MODEL}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Configure LoRA adapters
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./scbe-sft-output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    push_to_hub=True,
    hub_model_id=OUTPUT_MODEL,
    hub_strategy="every_save",
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,  # Pack short examples together for efficiency
)

print("Trainer ready. Starting training...")

In [ ]:
# TRAIN!
trainer.train()

print("\nTraining complete!")

In [ ]:
# Push final model to HuggingFace
trainer.push_to_hub()
print(f"\nModel pushed to: https://huggingface.co/{OUTPUT_MODEL}")

In [ ]:
# Quick test
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)

test_prompts = [
    "What is SCBE?",
    "Explain the 14-layer pipeline",
    "What are Sacred Tongues?",
]

for prompt in test_prompts:
    formatted_prompt = (
        f"<|system|>\nYou are an AI assistant powered by SCBE-AETHERMOORE governance.</s>\n"
        f"<|user|>\n{prompt}</s>\n"
        f"<|assistant|>\n"
    )
    result = pipe(formatted_prompt)
    print(f"\nQ: {prompt}")
    print(f"A: {result[0]['generated_text'].split('<|assistant|>')[-1][:300]}")
    print("-" * 60)

## Next Steps

1. **Upgrade base model**: Try `microsoft/phi-2` (2.7B) or `mistralai/Mistral-7B` for better quality
2. **More data**: Add Telegram conversations, combat blockchain data, web research pairs
3. **DPO training**: Use the chosen/rejected pairs for preference alignment
4. **Deploy**: Load the fine-tuned model in OctoArmor as a local tentacle
5. **Iterate**: Every interaction generates more training data via the flywheel